# CSCI 345 · Week 11 Lab 2 (Practice): Forecasting Bike-Share Demand

**Scenario.** A city bike-share company has to move bikes between stations by truck. To plan the trucks, they need to predict **how many bikes will be rented in the next hour**.

You'll follow **the same recipe as the electricity demo**, on a new dataset with some new twists:
- rentals are **counts** (they can't be negative), and many night-time hours are near zero;
- weekdays have **two commute peaks**, weekends have one **midday hump**;
- **rain** strongly suppresses rentals, and rain starts suddenly.

**How to work:** cells marked ✏️ **TODO** need your code (replace `raise NotImplementedError`). ✅ cells check your work. 📝 questions should be answered in a markdown cell. Use the demo notebook as your reference; most TODOs mirror a demo step.

**Deliverable:** this notebook with all cells run, plus answers to the 📝 questions (Part 8).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True})
PURPLE, GOLD, GRAY, RED = "#4B2E83", "#D4A017", "#888888", "#B01E2F"

### Toolkit (same RNN as the demo)
`make_bike_data()` generates the dataset if `bike_rentals.csv` isn't available. Like the demo, the data is **simulated** to behave like a real bike-share system.

In [ ]:
# ---------------- toolkit: data source + RNN model (run me) ----------------
def make_bike_data(n_days=365, seed=11):
    """Simulated hourly bike-share rentals for a city station network."""
    r = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_days * 24, freq="h")
    hour = idx.hour.values
    doy = idx.dayofyear.values
    weekend = (idx.dayofweek.values >= 5).astype(float)
    weather = np.zeros(len(idx))
    for i in range(1, len(idx)):
        weather[i] = 0.993 * weather[i - 1] + r.normal(0, 0.4)
    temp = 13 - 11 * np.cos(2 * np.pi * (doy - 15) / 365) + 5 * np.sin(2 * np.pi * (hour - 9) / 24) + weather
    # rain: storms that last a few hours
    rain = np.zeros(len(idx))
    i = 0
    while i < len(idx):
        if r.random() < 0.012:
            dur = r.integers(2, 10)
            rain[i:i + dur] = r.uniform(0.5, 6.0)
            i += dur
        else:
            i += 1
    commute = np.exp(-((hour - 8) ** 2) / 1.5) * 1.0 + np.exp(-((hour - 17.5) ** 2) / 2.5) * 1.2 + 0.25 * np.exp(-((hour - 13) ** 2) / 6)
    leisure = np.exp(-((hour - 14) ** 2) / 10) * 1.1
    shape = (1 - weekend) * commute + weekend * leisure + 0.03
    comfort = np.exp(-((temp - 23) ** 2) / 120)
    rain_effect = np.exp(-0.45 * rain)
    lam = 260 * shape * comfort * rain_effect + 3
    rentals = r.poisson(lam)
    return pd.DataFrame({"timestamp": idx, "rentals": rentals, "temperature_c": temp.round(1), "rain_mm": rain.round(1)})


class RNNRegressor:
    """A vanilla RNN (many-to-one) for regression, trained with BPTT + Adam. NumPy only."""

    def __init__(self, input_size, hidden_size=32, output_size=1, seed=0):
        r = np.random.default_rng(seed)
        H = hidden_size
        self.P = {"Wxh": r.normal(0, 1 / np.sqrt(input_size), (input_size, H)),
                  "Whh": r.normal(0, 1 / np.sqrt(H), (H, H)) * 0.9,
                  "bh": np.zeros(H),
                  "Why": r.normal(0, 1 / np.sqrt(H), (H, output_size)),
                  "by": np.zeros(output_size)}
        self.H = H

    def _forward(self, X):
        B, T, _ = X.shape
        hs = [np.zeros((B, self.H))]
        for t in range(T):
            hs.append(np.tanh(X[:, t] @ self.P["Wxh"] + hs[-1] @ self.P["Whh"] + self.P["bh"]))
        return hs, hs[-1] @ self.P["Why"] + self.P["by"]

    def predict(self, X, batch=2048):
        return np.concatenate([self._forward(X[i:i + batch])[1] for i in range(0, len(X), batch)])

    def _grads(self, X, Y):
        B, T, _ = X.shape
        hs, Yhat = self._forward(X)
        err = Yhat - Y
        loss = (err ** 2).mean()
        G = {k: np.zeros_like(v) for k, v in self.P.items()}
        dY = 2 * err / err.size
        G["Why"] = hs[-1].T @ dY
        G["by"] = dY.sum(0)
        dh = dY @ self.P["Why"].T
        for t in reversed(range(T)):
            draw = (1 - hs[t + 1] ** 2) * dh
            G["bh"] += draw.sum(0)
            G["Wxh"] += X[:, t].T @ draw
            G["Whh"] += hs[t].T @ draw
            dh = draw @ self.P["Whh"].T
        return loss, G

    def fit(self, X, Y, X_val, Y_val, epochs=30, batch=64, lr=3e-3, clip=1.0, patience=5, seed=0, verbose=True):
        r = np.random.default_rng(seed)
        m = {k: np.zeros_like(v) for k, v in self.P.items()}
        v = {k: np.zeros_like(v) for k, v in self.P.items()}
        t, best, best_P, wait = 0, np.inf, None, 0
        history = {"train": [], "val": []}
        for ep in range(1, epochs + 1):
            order = r.permutation(len(X))
            losses = []
            for i in range(0, len(X), batch):
                b = order[i:i + batch]
                loss, G = self._grads(X[b], Y[b])
                norm = np.sqrt(sum((g ** 2).sum() for g in G.values()))
                if norm > clip:                                   # clip by global norm
                    G = {k: g * clip / norm for k, g in G.items()}
                t += 1
                for k in self.P:                                  # Adam
                    m[k] = 0.9 * m[k] + 0.1 * G[k]
                    v[k] = 0.999 * v[k] + 0.001 * G[k] ** 2
                    self.P[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
                losses.append(loss)
            val = ((self.predict(X_val) - Y_val) ** 2).mean()
            history["train"].append(np.mean(losses)); history["val"].append(val)
            if verbose:
                print(f"epoch {ep:>2}  train MSE {np.mean(losses):.4f}  val MSE {val:.4f}" + ("  *" if val < best else ""))
            if val < best:
                best, best_P, wait = val, {k: p.copy() for k, p in self.P.items()}, 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose: print(f"early stopping: no improvement for {patience} epochs")
                    break
        self.P = best_P
        return history


def make_windows(features, target, window, horizon=1, first_row=1):
    """Slide a window over time.

    For each target hour i:
        X = features[i - window + 1 : i + 1]   (the `window` rows ending at hour i)
        Y = target[i : i + horizon]            (the value(s) we want to predict)
    Returns X (N, window, D), Y (N, horizon), and idx (the target hour of each sample).
    """
    idx = np.arange(first_row + window - 1, len(features) - horizon + 1)
    X = np.stack([features[i - window + 1: i + 1] for i in idx])
    Y = np.stack([target[i: i + horizon] for i in idx])
    return X, Y, idx

def mae(pred, true):
    return np.mean(np.abs(pred - true))

print('toolkit loaded')

---
## Part 1 · Load and explore (15 min)

In [ ]:
try:
    df = pd.read_csv("bike_rentals.csv", parse_dates=["timestamp"])
    print("Loaded bike_rentals.csv")
except FileNotFoundError:
    df = make_bike_data()
    print("CSV not found: generated the same dataset with make_bike_data()")
df["hour"] = df.timestamp.dt.hour
df["is_weekend"] = (df.timestamp.dt.dayofweek >= 5).astype(int)
print(df.shape)
df.head()

✏️ **TODO 1.** Make three plots:
1. rentals over the whole year,
2. the **average daily profile** for weekdays vs. weekends (hint: copy the demo's `groupby(["hour", "is_weekend"])` line),
3. average rentals when it's **raining** (`rain_mm > 0`) vs. **dry**, for daytime hours 7–20 only.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
# ✏️ YOUR CODE HERE (hint: three plots: ax[0] the series, ax[1] the daily profile by is_weekend, ax[2] a bar chart of rain vs. dry)
raise NotImplementedError("Complete this TODO")
plt.tight_layout(); plt.show()

📝 **Q1.** Describe the weekday and weekend patterns in one sentence each. Roughly how much does rain reduce daytime rentals?

---
## Part 2 · Features and windows (20 min)

✏️ **TODO 2.** Build the feature matrix. Each row (hour $t$) should contain:

| column | feature |
|---|---|
| 0 | rentals in the **previous** hour, normalized |
| 1 | temperature at hour $t$, normalized |
| 2 | rain at hour $t$, normalized |
| 3, 4 | sin and cos of the hour |
| 5 | is_weekend |

Use the split points below and **normalize with training statistics only**.

In [ ]:
n = len(df)
train_end, val_end = int(0.70 * n), int(0.85 * n)

y = df.rentals.values.astype(float)
temp = df.temperature_c.values.astype(float)
rain = df.rain_mm.values.astype(float)

# ✏️ YOUR CODE HERE (hint: compute y_mu, y_sd (and temp/rain stats) on [:train_end], build y_z, the lag, then np.column_stack)
raise NotImplementedError("Complete this TODO")
FEATURE_NAMES = ["rentals_lag1", "temperature", "rain", "hour_sin", "hour_cos", "is_weekend"]

In [ ]:
# ✅ check
assert features.shape == (n, 6), f"expected ({n}, 6), got {features.shape}"
assert np.isnan(features[0, 0]) and np.isclose(features[5, 0], y_z[4]), "column 0 must be the PREVIOUS hour's normalized rentals"
assert abs(features[:train_end, 1].mean()) < 1e-6, "normalize temperature with training statistics"
assert np.isclose(y_z[:train_end].mean(), 0) and np.isclose(y_z[:train_end].std(), 1), "normalize y with training statistics"
print("✅ features look right")
pd.DataFrame(features[1:6], columns=FEATURE_NAMES).round(2)

✏️ **TODO 3.** Use `make_windows` with a 24-hour window, then create the train/val/test masks **by time**, as in the demo.

In [ ]:
WINDOW = 24
# ✏️ YOUR CODE HERE (hint: X, Y, idx = make_windows(...); then boolean masks tr, va, te based on idx)
raise NotImplementedError("Complete this TODO")
X_tr, Y_tr, X_va, Y_va, X_te, Y_te = X[tr], Y[tr], X[va], Y[va], X[te], Y[te]

In [ ]:
# ✅ check
assert X.shape[1:] == (WINDOW, 6) and Y.shape[1] == 1
assert idx[tr].max() < idx[va].min() and idx[va].max() < idx[te].min(), "splits must be in time order, with no overlap"
assert not np.isnan(X).any(), "no NaNs should be inside any window"
for name, m in [("train", tr), ("val", va), ("test", te)]:
    print(f"{name:>5}: {m.sum():>5} samples   {df.timestamp[idx[m][0]]:%b %d} → {df.timestamp[idx[m][-1]]:%b %d}")
print("✅ windows and splits look right")

---
## Part 3 · Baselines (10 min)

✏️ **TODO 4.** Compute the test-set MAE (in bikes per hour) for **persistence** and **seasonal naive**. Store them in the `results` dict.

In [ ]:
y_true_te = y[idx[te]]
results = {}
# ✏️ YOUR CODE HERE (hint: same as the demo: y[idx[te] - 1] and y[idx[te] - 24])
raise NotImplementedError("Complete this TODO")
for k, v in results.items():
    print(f"{k:<40} MAE = {v:6.2f} bikes/hour")

📝 **Q2.** In the electricity demo, seasonal naive beat persistence. Which wins here? Why might "same hour yesterday" be a poor guess for bikes? *(Hint: think about Mondays, Saturdays and rainy days.)*

---
## Part 4 · Train the RNN (10 min)

✏️ **TODO 5.** Create an `RNNRegressor` (hidden size 32), train it with early stopping, and plot the learning curves.

In [ ]:
# ✏️ YOUR CODE HERE (hint: RNNRegressor(input_size=X.shape[2], hidden_size=32, output_size=1, seed=0); history = model.fit(...))
raise NotImplementedError("Complete this TODO")
plt.plot(history["train"], label="train", color=PURPLE); plt.plot(history["val"], label="validation", color=GOLD)
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.show()

---
## Part 5 · Evaluate (15 min)

✏️ **TODO 6.** Predict on the test set, convert back to bikes/hour, and **clip at 0** (you can't rent −3 bikes). Add the result to `results` and make the bar chart.

In [ ]:
# ✏️ YOUR CODE HERE (hint: model.predict(X_te)[:, 0] * y_sd + y_mu, then np.clip(..., 0, None))
raise NotImplementedError("Complete this TODO")
for k, v in results.items():
    print(f"{k:<40} MAE = {v:6.2f}")
plt.figure(figsize=(8, 3))
plt.barh(list(results.keys()), list(results.values()), color=[GRAY] * (len(results) - 1) + [PURPLE])
plt.gca().invert_yaxis(); plt.xlabel("test MAE (bikes/hour)"); plt.tight_layout(); plt.show()

In [ ]:
# ✅ check
assert pred_te.shape == y_true_te.shape and pred_te.min() >= 0
assert results["RNN (24-hour window)"] < min(results["Persistence (last hour)"], results["Seasonal naive (same hour yesterday)"]), \
    "Your RNN should beat both baselines. Check the normalization and the windows."
print("✅ The RNN beats both baselines")

✏️ **TODO 7.** Diagnose the errors. Plot one test week (actual vs. RNN vs. seasonal naive), then report the MAE separately for **weekdays vs. weekends** and for **rainy vs. dry** hours.

In [ ]:
# ✏️ YOUR CODE HERE (hint: reuse the demo's week plot; then build a DataFrame of errors with is_weekend and rain flags and groupby)
raise NotImplementedError("Complete this TODO")


📝 **Q3.** Where is the model weakest? Give one idea for an extra input feature that could help there.

---
## Part 6 · Experiments (20 min)

✏️ **TODO 8.** Write `train_and_score(feature_cols, window, hidden)`. It should rebuild the windows, train a fresh model and return the **validation** MAE (choose settings on validation, not test). Then run:
1. **Ablation:** all features vs. no rain (drop column 2) vs. no temperature (drop column 1).
2. **Window length:** 6, 12, 24, 48 hours.
3. **Hidden size:** 8, 32, 64.

In [ ]:
def train_and_score(feature_cols, window=24, hidden=32):
    # ✏️ YOUR CODE HERE (hint: like the demo's helper, but score on the VALIDATION mask (and clip predictions at 0))
    raise NotImplementedError("Complete this TODO")

ALL = [0, 1, 2, 3, 4, 5]
print("Ablation (validation MAE):")
for name, cols in [("all features", ALL), ("no rain", [0, 1, 3, 4, 5]), ("no temperature", [0, 2, 3, 4, 5])]:
    print(f"  {name:<15} {train_and_score(cols):.2f}")

In [ ]:
print("Window length (validation MAE):")
for w in [6, 12, 24, 48]:
    print(f"  {w:>3} h  {train_and_score(ALL, window=w):.2f}")
print("Hidden size (validation MAE):")
for h in [8, 32, 64]:
    print(f"  H = {h:>2}  {train_and_score(ALL, hidden=h):.2f}")

📝 **Q4.** Which feature matters most? Which window length and hidden size would you pick, and why? Did the bigger model always win?

---
## Part 7 · Challenge: plan the trucks for tomorrow (optional, ⭐⭐)

The operations team plans truck routes **once a day**, so they need all 24 hours at once. Build a **direct 24-output RNN** (as in demo Step 8) and compare its day-ahead MAE with seasonal naive. Then plot the MAE by hours-ahead.

In [ ]:
# ✏️ YOUR CODE HERE (hint: make_windows(..., horizon=24); RNNRegressor(..., output_size=24); evaluate one forecast per test day)
raise NotImplementedError("Complete this TODO")


---
## Part 8 · Wrap-up questions 📝

Answer Q1–Q4 above, plus:

**Q5.** Our test set covers mid-November to December. Why are the MAE numbers here not directly comparable to a summer test period? How would you make a fairer evaluation? *(Hint: look up "rolling-origin" or "walk-forward" evaluation.)*

**Q6.** A teammate shuffles all the windows randomly before splitting into train/val/test and reports a much better MAE. Explain to them what went wrong.

**Q7.** The rain feature uses *this hour's* rain. In real deployment, where would that number come from, and what happens to accuracy if the weather forecast is wrong?

### More practice (pick any)
- ⭐ Add a **lag-24** feature (rentals at the same hour yesterday) as an extra column. Does it help?
- ⭐⭐ Implement the **recursive** day-ahead forecast from demo Step 8 for bikes and compare it with your direct model. Does feeding predictions back in work as well here as it did for electricity? Why or why not? *(Hint: rain.)*
- ⭐ Replace MAE with **RMSE**. Which hours dominate RMSE, and why?
- ⭐⭐ Swap the vanilla RNN for an **LSTM in PyTorch** (on Colab) using the same windows, and compare.
- ⭐⭐ Use the **energy dataset** from the demo, but predict **3 hours ahead** with a direct model. How much worse is it than 1 hour ahead?
- ⭐⭐⭐ Implement **walk-forward evaluation**: retrain each month on all data up to that month, test on the next month, and plot MAE by month.